# Сборка датасета

In [1]:
from onetrans.ext.yambda.datacookin import DataCookinYambdaRank
from onetrans.run.config import dataset_config
from onetrans.ext.yambda.dataset import BinaryRankinArchive


cookin = DataCookinYambdaRank()
train_set, test_set = cookin.run(dataset_config)

/Users/oleg/projects/OneTrans_HSE_project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from onetrans.ext.yambda.dataset import BinaryRankinArchive

listens, timestamp_test_start = cookin.cook(dataset_config)
archive = BinaryRankinArchive(listens)

In [3]:
meta = archive.meta

In [4]:
import polars as pl
train_listens = listens.filter(pl.col('timestamp') < timestamp_test_start)

In [5]:
from onetrans.run.config import DENSE_COLUMNS
train_listens_dense_million = train_listens[DENSE_COLUMNS].slice(0, 1_000_000)

In [6]:
batch = next(iter(train_set))

In [7]:
from onetrans.baselines.dcn_v2 import DCNV2

model = DCNV2(
    embedding_size=64,
    cross_layers=6,
    deep_units=[256, 128, 64],
    input_size=587,
    dense_train_df=train_listens_dense_million,
    n_bins=32,
    output_size=1,
    train_df_slice=1_000_000,
    num_albums=meta['num_albums'],
    num_artists=meta['num_artists'],
    num_users=meta['num_users'],
    num_items=meta['num_items']
)

In [8]:
import gc

del train_listens
del listens
del train_listens_dense_million
del archive
gc.collect()

564

In [9]:
batch['NS']['sparse_features']['item_id']

tensor([107808,  79748, 173751, 101758, 104247, 146681, 114974, 113254])

In [10]:
batch

{'S': {'item_id': tensor([ 44460,  39928, 117196, 185506,  67432, 137447,  31472, 195765,  60987,
          177563, 190137, 106113, 127828, 109534,  32312, 122244, 115952, 177760,
           85290,  10705, 166363,  33399, 138552, 124956, 161327, 117954,  41679,
          147645, 100521, 210199, 144757, 152305,  13214,  73801, 138015, 135043,
          167930,  55315,  11984,  65395,  41132, 144163,  85670,  79754, 113048,
          191531, 106826,  66449, 200778,  23574, 127333, 127333,  56690,  63333,
            7240,  67020, 136949, 152828, 126777, 144432,  65463, 197118, 202208,
          185929, 201894,  56918, 181471, 153373, 168900, 173229, 110760,  69333,
           28851, 168710, 188933, 158321,  10468, 100537,  66618, 184266,  92884,
           18760, 198290,  83724,  78505, 168181,  98434, 191594,  82374, 186220,
           45158,  12016, 203923,  59795, 188239,  18489,  22074, 161737,  50446,
          214083, 109506, 111681, 156911, 210800,  39737,  86271, 166215,  66635,


In [28]:
from onetrans.nn.encoders.categorial import CategoricalEncoder
from torch import nn

embedding = nn.Embedding(
    num_embeddings=65536,
    embedding_dim=64
)
categorical_encoder = CategoricalEncoder(embedding)

In [11]:
model.item_encoder(batch['NS']['sparse_features']['item_id'])

tensor([[-7.6313e-02, -7.7335e-01, -4.2124e-01,  3.2569e-01, -4.5934e-01,
          2.8563e+00,  1.1418e+00, -1.3037e-01, -1.0133e+00, -3.5837e+00,
         -1.4521e+00,  1.1447e+00, -9.6171e-01,  2.6721e-01, -6.0086e-02,
          2.4246e-01,  1.8187e+00,  6.1977e-01, -1.3667e+00,  1.7351e+00,
          8.7015e-01,  7.1678e-01,  1.4914e+00, -2.4722e-01,  1.2308e-01,
          1.3726e+00, -9.3230e-01, -2.7456e-02, -3.7130e-02,  1.0029e+00,
          2.9474e-01,  5.2142e-01,  9.2638e-01,  1.1399e+00,  1.1816e+00,
         -5.1196e-01, -7.2770e-01,  5.8463e-01, -1.8697e+00,  1.1424e+00,
          5.4580e-02,  1.9536e+00,  9.3859e-01,  1.2931e+00,  8.5732e-01,
         -7.2826e-02,  1.7035e+00, -9.9637e-01, -1.4365e-01, -6.5964e-01,
         -1.4614e+00, -1.5973e-01,  1.9131e-01, -8.6610e-02,  1.1514e+00,
          1.5497e+00, -3.9685e-01,  2.2662e-01,  5.5555e-01,  9.0799e-01,
         -5.8898e-01,  5.0202e-02, -5.1598e-01,  1.4031e+00],
        [ 3.9832e-01, -4.1234e-01,  1.5673e+00,  2

In [12]:
batch['NS']["multivalent_features"]["artist_ids"]["values"].shape

torch.Size([8])

In [13]:
batch['NS']["multivalent_features"]["artist_ids"]["lengths"].shape

torch.Size([8])

In [14]:
model.artist_encoder(
    batch['NS']["multivalent_features"]["artist_ids"]["values"],
    batch['NS']["multivalent_features"]["artist_ids"]["lengths"]
)

/Users/oleg/projects/OneTrans_HSE_project/onetrans/nn/encoders/multivalent.py:20: UserWarning: Argument order of nn.functional.embedding_bag was changed. Usage `embedding_bag(weight, input, ...)` is deprecated, and should now be `embedding_bag(input, weight, ...)`.
  emb = embedding_bag(


tensor([[-8.2663e-01,  3.3806e-01, -2.4042e-01,  1.5134e-01, -1.5777e+00,
          7.4039e-02,  9.6793e-02,  3.6227e-01, -3.3259e-01, -7.2329e-01,
          4.7080e-01, -1.2249e+00, -4.2436e-01, -8.5009e-02, -4.1618e-01,
         -2.6779e-01,  1.9072e+00,  4.9037e-01, -1.3363e+00,  6.1008e-01,
         -1.0459e+00, -2.7494e-01,  8.5681e-03,  1.4889e+00,  1.8598e-01,
          7.8682e-01, -1.6864e-01, -4.2594e-01,  1.9290e+00,  1.3341e+00,
          1.3237e+00, -5.5245e-01,  9.9686e-01, -1.2722e+00,  1.0637e+00,
          2.1026e+00, -3.3596e+00,  4.9999e-01,  2.8872e-01,  7.4114e-01,
          1.6953e-01,  6.3310e-01,  5.1021e-01, -6.6536e-01, -1.6206e+00,
         -4.9791e-01,  1.3179e+00, -1.4866e+00, -1.9585e-01,  7.9347e-02,
         -1.6311e+00, -8.3995e-01,  1.5812e-01, -6.8630e-02, -4.6003e-01,
         -1.0052e+00,  6.4516e-01,  1.0599e+00, -6.7050e-01, -6.7435e-01,
         -1.7243e+00, -1.5566e+00, -2.5825e+00, -7.4733e-02],
        [ 1.2464e+00,  2.2239e+00, -7.8817e-01,  1

In [15]:
model(batch['NS'])

tensor([[0.1234],
        [0.3654],
        [0.3751],
        [0.4478],
        [0.1163],
        [0.4010],
        [0.3436],
        [0.1847]], grad_fn=<AddmmBackward0>)